In [1]:
# =========================================================
# INSTALL DEPENDENCIES
# =========================================================
!pip install -q transformers datasets evaluate jiwer accelerate soundfile huggingface_hub librosa

# =========================================================
# IMPORTS
# =========================================================
import os
import random
import numpy as np
import torch
import pandas as pd
import evaluate
import librosa

from huggingface_hub import login

from dataclasses import dataclass
from typing import Any

from datasets import Dataset, Audio
from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    EarlyStoppingCallback,
)

# =========================================================
# HF LOGIN
# Prompts you to paste your token at runtime (hidden input, like a
# PIN) instead of hardcoding it in the script.
# =========================================================
from getpass import getpass

login(getpass("Enter your Hugging Face token: "))

# =========================================================
# DEVICE
# =========================================================
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# =========================================================
# PATHS
# =========================================================
train_csv = "SPCS/manifests/train.csv"
valid_csv = "SPCS/manifests/valid.csv"

# =========================================================
# LOAD DATA
# =========================================================
train_df = pd.read_csv(train_csv)
valid_df = pd.read_csv(valid_csv)

print("Training samples:", len(train_df))
print("Validation samples:", len(valid_df))

# =========================================================
# DATASET CONVERSION
# =========================================================
train_ds = Dataset.from_pandas(train_df)
valid_ds = Dataset.from_pandas(valid_df)

train_ds = train_ds.cast_column("audio", Audio(sampling_rate=16000))
valid_ds = valid_ds.cast_column("audio", Audio(sampling_rate=16000))

# =========================================================
# MODEL + PROCESSOR
# Switch model_name to "openai/whisper-tiny" if small keeps
# overfitting hard even after the regularization changes below.
# =========================================================
model_name = "openai/whisper-tiny"

processor = WhisperProcessor.from_pretrained(model_name, task="transcribe")
model = WhisperForConditionalGeneration.from_pretrained(model_name)

model.generation_config.forced_decoder_ids = None
model.generation_config.suppress_tokens = []
model.generation_config.language = None
model.generation_config.task = "transcribe"

# ---- Regularization additions ----
# SpecAugment: randomly masks time/frequency bands in the input
# spectrogram during training, forcing the model to not rely on
# any single narrow slice of audio. Cheap and effective against
# overfitting on small speech datasets.
model.config.apply_spec_augment = True
model.config.mask_time_prob = 0.05
model.config.mask_time_length = 10
model.config.mask_feature_prob = 0.05
model.config.mask_feature_length = 10

# Optional: freeze the encoder if your training set is very small
# (e.g. a few hours of audio). This forces the model to only adapt
# the decoder/language side, which is much less prone to overfitting
# on limited data. Uncomment if train loss still crashes to ~0 fast.
# model.freeze_encoder()

model.to(device)

# =========================================================
# SPEED AUGMENTATION (new)
# Randomly speeds up or slows down training audio before feature
# extraction. This is a standard, cheap augmentation for ASR
# (commonly called "speed perturbation") that exposes the model to
# faster/slower speaking rates, helping it generalize better and
# reducing overfitting on a small dataset. Applied to TRAINING data
# only - validation audio must stay unmodified so your WER/CER stay
# a fair, untouched measure of real performance.
# =========================================================
SPEED_FACTORS = [0.9, 1.0, 1.1]  # 0.9 = slower, 1.1 = faster, 1.0 = unchanged

def apply_speed_augmentation(audio_array, sampling_rate):
    speed_factor = random.choice(SPEED_FACTORS)
    if speed_factor == 1.0:
        return audio_array
    # librosa.effects.time_stretch changes playback speed (duration)
    # while keeping the audio otherwise intact.
    return librosa.effects.time_stretch(audio_array.astype(np.float32), rate=speed_factor)

# =========================================================
# PREPROCESSING
# =========================================================
def prepare_dataset_train(batch):
    audio = batch["audio"]

    audio_array = apply_speed_augmentation(audio["array"], audio["sampling_rate"])

    batch["input_features"] = processor.feature_extractor(
        audio_array,
        sampling_rate=audio["sampling_rate"]
    ).input_features[0]

    batch["labels"] = processor.tokenizer(batch["text"]).input_ids

    return batch

def prepare_dataset_eval(batch):
    audio = batch["audio"]

    batch["input_features"] = processor.feature_extractor(
        audio["array"],
        sampling_rate=audio["sampling_rate"]
    ).input_features[0]

    batch["labels"] = processor.tokenizer(batch["text"]).input_ids

    return batch

train_ds = train_ds.map(
    prepare_dataset_train,
    remove_columns=train_ds.column_names,
    num_proc=1
)

valid_ds = valid_ds.map(
    prepare_dataset_eval,
    remove_columns=valid_ds.column_names,
    num_proc=1
)

# =========================================================
# DATA COLLATOR
# =========================================================
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features):
        input_features = [{"input_features": f["input_features"]} for f in features]

        batch = self.processor.feature_extractor.pad(
            input_features,
            return_tensors="pt"
        )

        label_features = [{"input_ids": f["labels"]} for f in features]

        labels_batch = self.processor.tokenizer.pad(
            label_features,
            return_tensors="pt"
        )

        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1),
            -100
        )

        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor)

# =========================================================
# WER + CER METRICS
# =========================================================
wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str = processor.tokenizer.batch_decode(
        pred_ids,
        skip_special_tokens=True
    )

    label_str = processor.tokenizer.batch_decode(
        label_ids,
        skip_special_tokens=True
    )

    wer = wer_metric.compute(
        predictions=pred_str,
        references=label_str
    )

    cer = cer_metric.compute(
        predictions=pred_str,
        references=label_str
    )

    return {
        "wer": wer,
        "cer": cer
    }

# =========================================================
# TRAINING CONFIG (FIXED - anti-overfitting)
# =========================================================
training_args = Seq2SeqTrainingArguments(
    output_dir="SPCS/models_result/whisper_speed_aug",

    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,   # effective batch size 8

    learning_rate=1e-5,
    lr_scheduler_type="linear",
    warmup_ratio=0.1,               # smooth ramp-up instead of jumping straight to peak LR
    weight_decay=0.01,              # penalizes large weights, directly fights overfitting

    num_train_epochs=10,            # kept high on purpose - early stopping will cut it short
    max_grad_norm=1.0,

    fp16=torch.cuda.is_available(),
    gradient_checkpointing=True,

    eval_strategy="epoch",
    save_strategy="epoch",

    logging_steps=25,
    save_total_limit=3,

    predict_with_generate=True,
    generation_max_length=225,

    # Keep the checkpoint with the best validation WER, not the last one.
    # This alone fixes most of the damage from overfitting: even if
    # training runs long, you deploy the epoch that generalized best.
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,

    # NOTE: safe_serialization is handled manually at save time below
    # (this transformers version doesn't accept save_safetensors here).

    report_to="none"
)

# =========================================================
# TRAINER
# =========================================================
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=valid_ds,
    data_collator=data_collator,
    processing_class=processor.feature_extractor,
    compute_metrics=compute_metrics,
    # Stops training if validation WER hasn't improved for 3 straight
    # evals - prevents the "train loss -> 0, val loss flat" pattern
    # you saw from running all the way to epoch 10.
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

# =========================================================
# TRAIN
# =========================================================
trainer.train()

# =========================================================
# SAVE MODEL (this saves the BEST checkpoint, thanks to
# load_best_model_at_end=True above)
# =========================================================
save_path = "SPCS/models_result/whisper_speed_aug"

model.tie_weights()  # make sure proj_out is explicitly linked to the embedding weights

# Saving directly via model.save_pretrained (rather than trainer.save_model)
# so we can pass safe_serialization=False - this avoids the "missing keys:
# proj_out.weight" warning caused by Whisper's tied embedding/output layer
# being dropped from safetensors checkpoints on save.
model.save_pretrained(save_path, safe_serialization=False)
processor.save_pretrained(save_path)

print("\nTraining complete.")
print("Best model (by validation WER) saved to:", save_path)

Enter your Hugging Face token:  ········


Using device: cuda
Training samples: 9908
Validation samples: 1239


Loading weights:   0%|          | 0/167 [00:00<?, ?it/s]

Map:   0%|          | 0/9908 [00:00<?, ? examples/s]

Map:   0%|          | 0/1239 [00:00<?, ? examples/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['proj_out.weight'].


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Training complete.
Best model (by validation WER) saved to: SPCS/models_result/whisper_speed_aug


In [2]:
# =========================================================
# INSTALL DEPENDENCIES
# =========================================================
!pip install -q transformers datasets evaluate jiwer accelerate soundfile huggingface_hub librosa

# =========================================================
# IMPORTS
# =========================================================
import os
import random
import numpy as np
import torch
import pandas as pd
import evaluate
import librosa

from huggingface_hub import login

from dataclasses import dataclass
from typing import Any

from datasets import Dataset, Audio
from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    EarlyStoppingCallback,
)

# =========================================================
# HF LOGIN
# Prompts you to paste your token at runtime (hidden input, like a
# PIN) instead of hardcoding it in the script.
# =========================================================
from getpass import getpass

login(getpass("Enter your Hugging Face token: "))

# =========================================================
# DEVICE
# =========================================================
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# =========================================================
# PATHS
# =========================================================
train_csv = "SPCS/manifests/train.csv"
valid_csv = "SPCS/manifests/valid.csv"

# =========================================================
# LOAD DATA
# =========================================================
train_df = pd.read_csv(train_csv)
valid_df = pd.read_csv(valid_csv)

print("Training samples:", len(train_df))
print("Validation samples:", len(valid_df))

# =========================================================
# DATASET CONVERSION
# =========================================================
train_ds = Dataset.from_pandas(train_df)
valid_ds = Dataset.from_pandas(valid_df)

train_ds = train_ds.cast_column("audio", Audio(sampling_rate=16000))
valid_ds = valid_ds.cast_column("audio", Audio(sampling_rate=16000))

# =========================================================
# MODEL + PROCESSOR
# Switch model_name to "openai/whisper-tiny" if small keeps
# overfitting hard even after the regularization changes below.
# =========================================================
model_name = "openai/whisper-tiny"

processor = WhisperProcessor.from_pretrained(model_name, task="transcribe")
model = WhisperForConditionalGeneration.from_pretrained(model_name)

model.generation_config.forced_decoder_ids = None
model.generation_config.suppress_tokens = []
model.generation_config.language = None
model.generation_config.task = "transcribe"

# ---- Regularization additions ----
# SpecAugment: randomly masks time/frequency bands in the input
# spectrogram during training, forcing the model to not rely on
# any single narrow slice of audio. Cheap and effective against
# overfitting on small speech datasets.
model.config.apply_spec_augment = True
model.config.mask_time_prob = 0.05
model.config.mask_time_length = 10
model.config.mask_feature_prob = 0.05
model.config.mask_feature_length = 10

# Optional: freeze the encoder if your training set is very small
# (e.g. a few hours of audio). This forces the model to only adapt
# the decoder/language side, which is much less prone to overfitting
# on limited data. Uncomment if train loss still crashes to ~0 fast.
# model.freeze_encoder()

model.to(device)

# =========================================================
# PITCH AUGMENTATION
# Randomly shifts training audio up or down in pitch (in semitones)
# before feature extraction. This is a standard, cheap augmentation
# for ASR that exposes the model to voice-pitch variation, helping
# it generalize better and reducing overfitting on a small dataset.
# Applied to TRAINING data only - validation audio must stay
# unmodified so your WER/CER stay a fair, untouched measure of real
# performance.
# =========================================================
PITCH_STEPS = [-2, -1, 0, 1, 2]  # semitones: negative = lower, positive = higher, 0 = unchanged

def apply_pitch_augmentation(audio_array, sampling_rate):
    n_steps = random.choice(PITCH_STEPS)
    if n_steps == 0:
        return audio_array
    # librosa.effects.pitch_shift changes pitch while keeping
    # duration otherwise intact.
    return librosa.effects.pitch_shift(
        audio_array.astype(np.float32),
        sr=sampling_rate,
        n_steps=n_steps
    )

# =========================================================
# PREPROCESSING
# =========================================================
def prepare_dataset_train(batch):
    audio = batch["audio"]

    audio_array = apply_pitch_augmentation(audio["array"], audio["sampling_rate"])

    batch["input_features"] = processor.feature_extractor(
        audio_array,
        sampling_rate=audio["sampling_rate"]
    ).input_features[0]

    batch["labels"] = processor.tokenizer(batch["text"]).input_ids

    return batch

def prepare_dataset_eval(batch):
    audio = batch["audio"]

    batch["input_features"] = processor.feature_extractor(
        audio["array"],
        sampling_rate=audio["sampling_rate"]
    ).input_features[0]

    batch["labels"] = processor.tokenizer(batch["text"]).input_ids

    return batch

train_ds = train_ds.map(
    prepare_dataset_train,
    remove_columns=train_ds.column_names,
    num_proc=1
)

valid_ds = valid_ds.map(
    prepare_dataset_eval,
    remove_columns=valid_ds.column_names,
    num_proc=1
)

# =========================================================
# DATA COLLATOR
# =========================================================
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features):
        input_features = [{"input_features": f["input_features"]} for f in features]

        batch = self.processor.feature_extractor.pad(
            input_features,
            return_tensors="pt"
        )

        label_features = [{"input_ids": f["labels"]} for f in features]

        labels_batch = self.processor.tokenizer.pad(
            label_features,
            return_tensors="pt"
        )

        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1),
            -100
        )

        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor)

# =========================================================
# WER + CER METRICS
# =========================================================
wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str = processor.tokenizer.batch_decode(
        pred_ids,
        skip_special_tokens=True
    )

    label_str = processor.tokenizer.batch_decode(
        label_ids,
        skip_special_tokens=True
    )

    wer = wer_metric.compute(
        predictions=pred_str,
        references=label_str
    )

    cer = cer_metric.compute(
        predictions=pred_str,
        references=label_str
    )

    return {
        "wer": wer,
        "cer": cer
    }

# =========================================================
# TRAINING CONFIG (FIXED - anti-overfitting)
# =========================================================
training_args = Seq2SeqTrainingArguments(
    output_dir="SPCS/models_result/whisper_pitch_aug",

    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,   # effective batch size 8

    learning_rate=1e-5,
    lr_scheduler_type="linear",
    warmup_ratio=0.1,               # smooth ramp-up instead of jumping straight to peak LR
    weight_decay=0.01,              # penalizes large weights, directly fights overfitting

    num_train_epochs=10,            # kept high on purpose - early stopping will cut it short
    max_grad_norm=1.0,

    fp16=torch.cuda.is_available(),
    gradient_checkpointing=True,

    eval_strategy="epoch",
    save_strategy="epoch",

    logging_steps=25,
    save_total_limit=3,

    predict_with_generate=True,
    generation_max_length=225,

    # Keep the checkpoint with the best validation WER, not the last one.
    # This alone fixes most of the damage from overfitting: even if
    # training runs long, you deploy the epoch that generalized best.
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,

    # NOTE: safe_serialization is handled manually at save time below
    # (this transformers version doesn't accept save_safetensors here).

    report_to="none"
)

# =========================================================
# TRAINER
# =========================================================
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=valid_ds,
    data_collator=data_collator,
    processing_class=processor.feature_extractor,
    compute_metrics=compute_metrics,
    # Stops training if validation WER hasn't improved for 3 straight
    # evals - prevents the "train loss -> 0, val loss flat" pattern
    # you saw from running all the way to epoch 10.
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

# =========================================================
# TRAIN
# =========================================================
trainer.train()

# =========================================================
# SAVE MODEL (this saves the BEST checkpoint, thanks to
# load_best_model_at_end=True above)
# =========================================================
save_path = "SPCS/models_result/whisper_pitch_aug"

model.tie_weights()  # make sure proj_out is explicitly linked to the embedding weights

# Saving directly via model.save_pretrained (rather than trainer.save_model)
# so we can pass safe_serialization=False - this avoids the "missing keys:
# proj_out.weight" warning caused by Whisper's tied embedding/output layer
# being dropped from safetensors checkpoints on save.
model.save_pretrained(save_path, safe_serialization=False)
processor.save_pretrained(save_path)

print("\nTraining complete.")
print("Best model (by validation WER) saved to:", save_path)

Enter your Hugging Face token:  ········


Using device: cuda
Training samples: 9908
Validation samples: 1239


Loading weights:   0%|          | 0/167 [00:00<?, ?it/s]

Map:   0%|          | 0/9908 [00:00<?, ? examples/s]

Map:   0%|          | 0/1239 [00:00<?, ? examples/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['proj_out.weight'].


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Training complete.
Best model (by validation WER) saved to: SPCS/models_result/whisper_pitch_aug


In [3]:
# =========================================================
# INSTALL DEPENDENCIES
# =========================================================
!pip install -q transformers datasets evaluate jiwer accelerate soundfile huggingface_hub librosa

# =========================================================
# IMPORTS
# =========================================================
import os
import random
import numpy as np
import torch
import pandas as pd
import evaluate
import librosa

from huggingface_hub import login

from dataclasses import dataclass
from typing import Any

from datasets import Dataset, Audio
from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    EarlyStoppingCallback,
)

# =========================================================
# HF LOGIN
# Prompts you to paste your token at runtime (hidden input, like a
# PIN) instead of hardcoding it in the script.
# =========================================================
from getpass import getpass

login(getpass("Enter your Hugging Face token: "))

# =========================================================
# DEVICE
# =========================================================
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# =========================================================
# PATHS
# =========================================================
train_csv = "SPCS/manifests/train.csv"
valid_csv = "SPCS/manifests/valid.csv"

# =========================================================
# LOAD DATA
# =========================================================
train_df = pd.read_csv(train_csv)
valid_df = pd.read_csv(valid_csv)

print("Training samples:", len(train_df))
print("Validation samples:", len(valid_df))

# =========================================================
# DATASET CONVERSION
# =========================================================
train_ds = Dataset.from_pandas(train_df)
valid_ds = Dataset.from_pandas(valid_df)

train_ds = train_ds.cast_column("audio", Audio(sampling_rate=16000))
valid_ds = valid_ds.cast_column("audio", Audio(sampling_rate=16000))

# =========================================================
# MODEL + PROCESSOR
# Switch model_name to "openai/whisper-tiny" if small keeps
# overfitting hard even after the regularization changes below.
# =========================================================
model_name = "openai/whisper-tiny"

processor = WhisperProcessor.from_pretrained(model_name, task="transcribe")
model = WhisperForConditionalGeneration.from_pretrained(model_name)

model.generation_config.forced_decoder_ids = None
model.generation_config.suppress_tokens = []
model.generation_config.language = None
model.generation_config.task = "transcribe"

# ---- Regularization additions ----
# SpecAugment: randomly masks time/frequency bands in the input
# spectrogram during training, forcing the model to not rely on
# any single narrow slice of audio. Cheap and effective against
# overfitting on small speech datasets.
model.config.apply_spec_augment = True
model.config.mask_time_prob = 0.05
model.config.mask_time_length = 10
model.config.mask_feature_prob = 0.05
model.config.mask_feature_length = 10

# Optional: freeze the encoder if your training set is very small
# (e.g. a few hours of audio). This forces the model to only adapt
# the decoder/language side, which is much less prone to overfitting
# on limited data. Uncomment if train loss still crashes to ~0 fast.
# model.freeze_encoder()

model.to(device)

# =========================================================
# NOISE AUGMENTATION
# Randomly adds low-level Gaussian noise to training audio before
# feature extraction. This is a standard, cheap augmentation for ASR
# that exposes the model to noisy input conditions, helping it
# generalize better and reducing overfitting on a small dataset.
# Applied to TRAINING data only - validation audio must stay
# unmodified so your WER/CER stay a fair, untouched measure of real
# performance.
# =========================================================


NOISE_FACTORS = [0.0, 0.002, 0.005, 0.01]

def apply_noise_augmentation(audio_array, sampling_rate):
    noise_factor = random.choice(NOISE_FACTORS)
    if noise_factor == 0.0:
        return audio_array
    audio_array = audio_array.astype(np.float32)
    noise = np.random.randn(len(audio_array)).astype(np.float32)
    return audio_array + noise_factor * np.max(np.abs(audio_array)) * noise

# =========================================================
# PREPROCESSING
# =========================================================
def prepare_dataset_train(batch):
    audio = batch["audio"]

    audio_array = apply_noise_augmentation(audio["array"], audio["sampling_rate"])

    batch["input_features"] = processor.feature_extractor(
        audio_array,
        sampling_rate=audio["sampling_rate"]
    ).input_features[0]

    batch["labels"] = processor.tokenizer(batch["text"]).input_ids

    return batch

def prepare_dataset_eval(batch):
    audio = batch["audio"]

    batch["input_features"] = processor.feature_extractor(
        audio["array"],
        sampling_rate=audio["sampling_rate"]
    ).input_features[0]

    batch["labels"] = processor.tokenizer(batch["text"]).input_ids

    return batch

train_ds = train_ds.map(
    prepare_dataset_train,
    remove_columns=train_ds.column_names,
    num_proc=1
)

valid_ds = valid_ds.map(
    prepare_dataset_eval,
    remove_columns=valid_ds.column_names,
    num_proc=1
)

# =========================================================
# DATA COLLATOR
# =========================================================
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features):
        input_features = [{"input_features": f["input_features"]} for f in features]

        batch = self.processor.feature_extractor.pad(
            input_features,
            return_tensors="pt"
        )

        label_features = [{"input_ids": f["labels"]} for f in features]

        labels_batch = self.processor.tokenizer.pad(
            label_features,
            return_tensors="pt"
        )

        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1),
            -100
        )

        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor)

# =========================================================
# WER + CER METRICS
# =========================================================
wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str = processor.tokenizer.batch_decode(
        pred_ids,
        skip_special_tokens=True
    )

    label_str = processor.tokenizer.batch_decode(
        label_ids,
        skip_special_tokens=True
    )

    wer = wer_metric.compute(
        predictions=pred_str,
        references=label_str
    )

    cer = cer_metric.compute(
        predictions=pred_str,
        references=label_str
    )

    return {
        "wer": wer,
        "cer": cer
    }

# =========================================================
# TRAINING CONFIG (FIXED - anti-overfitting)
# =========================================================
training_args = Seq2SeqTrainingArguments(
    output_dir="SPCS/models_result/whisper_noise_aug",

    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,   # effective batch size 8

    learning_rate=1e-5,
    lr_scheduler_type="linear",
    warmup_ratio=0.1,               # smooth ramp-up instead of jumping straight to peak LR
    weight_decay=0.01,              # penalizes large weights, directly fights overfitting

    num_train_epochs=10,            # kept high on purpose - early stopping will cut it short
    max_grad_norm=1.0,

    fp16=torch.cuda.is_available(),
    gradient_checkpointing=True,

    eval_strategy="epoch",
    save_strategy="epoch",

    logging_steps=25,
    save_total_limit=3,

    predict_with_generate=True,
    generation_max_length=225,

    # Keep the checkpoint with the best validation WER, not the last one.
    # This alone fixes most of the damage from overfitting: even if
    # training runs long, you deploy the epoch that generalized best.
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,

    # NOTE: safe_serialization is handled manually at save time below
    # (this transformers version doesn't accept save_safetensors here).

    report_to="none"
)

# =========================================================
# TRAINER
# =========================================================
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=valid_ds,
    data_collator=data_collator,
    processing_class=processor.feature_extractor,
    compute_metrics=compute_metrics,
    # Stops training if validation WER hasn't improved for 3 straight
    # evals - prevents the "train loss -> 0, val loss flat" pattern
    # you saw from running all the way to epoch 10.
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

# =========================================================
# TRAIN
# =========================================================
trainer.train()

# =========================================================
# SAVE MODEL (this saves the BEST checkpoint, thanks to
# load_best_model_at_end=True above)
# =========================================================
save_path = "SPCS/models_result/whisper_noise_aug"

model.tie_weights()  # make sure proj_out is explicitly linked to the embedding weights

# Saving directly via model.save_pretrained (rather than trainer.save_model)
# so we can pass safe_serialization=False - this avoids the "missing keys:
# proj_out.weight" warning caused by Whisper's tied embedding/output layer
# being dropped from safetensors checkpoints on save.
model.save_pretrained(save_path, safe_serialization=False)
processor.save_pretrained(save_path)

print("\nTraining complete.")
print("Best model (by validation WER) saved to:", save_path)

Enter your Hugging Face token:  ········


Using device: cuda
Training samples: 9908
Validation samples: 1239


Loading weights:   0%|          | 0/167 [00:00<?, ?it/s]

Map:   0%|          | 0/9908 [00:00<?, ? examples/s]

Map:   0%|          | 0/1239 [00:00<?, ? examples/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['proj_out.weight'].


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Training complete.
Best model (by validation WER) saved to: SPCS/models_result/whisper_noise_aug


In [4]:
# =========================================================
# INSTALL DEPENDENCIES
# =========================================================
!pip install -q transformers datasets evaluate jiwer accelerate soundfile huggingface_hub librosa

# =========================================================
# IMPORTS
# =========================================================
import os
import random
import numpy as np
import torch
import pandas as pd
import evaluate
import librosa

from huggingface_hub import login

from dataclasses import dataclass
from typing import Any

from datasets import Dataset, Audio
from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    EarlyStoppingCallback,
)

# =========================================================
# HF LOGIN
# Prompts you to paste your token at runtime (hidden input, like a
# PIN) instead of hardcoding it in the script.
# =========================================================
from getpass import getpass

login(getpass("Enter your Hugging Face token: "))

# =========================================================
# DEVICE
# =========================================================
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# =========================================================
# PATHS
# =========================================================
train_csv = "SPCS/manifests/train.csv"
valid_csv = "SPCS/manifests/valid.csv"

# =========================================================
# LOAD DATA
# =========================================================
train_df = pd.read_csv(train_csv)
valid_df = pd.read_csv(valid_csv)

print("Training samples:", len(train_df))
print("Validation samples:", len(valid_df))

# =========================================================
# DATASET CONVERSION
# =========================================================
train_ds = Dataset.from_pandas(train_df)
valid_ds = Dataset.from_pandas(valid_df)

train_ds = train_ds.cast_column("audio", Audio(sampling_rate=16000))
valid_ds = valid_ds.cast_column("audio", Audio(sampling_rate=16000))

# =========================================================
# MODEL + PROCESSOR
# Switch model_name to "openai/whisper-tiny" if small keeps
# overfitting hard even after the regularization changes below.
# =========================================================
model_name = "openai/whisper-tiny"

processor = WhisperProcessor.from_pretrained(model_name, task="transcribe")
model = WhisperForConditionalGeneration.from_pretrained(model_name)

model.generation_config.forced_decoder_ids = None
model.generation_config.suppress_tokens = []
model.generation_config.language = None
model.generation_config.task = "transcribe"

# ---- Regularization additions ----
# SpecAugment: randomly masks time/frequency bands in the input
# spectrogram during training, forcing the model to not rely on
# any single narrow slice of audio. Cheap and effective against
# overfitting on small speech datasets. This is the ONLY augmentation
# in this version (no speed/pitch/noise perturbation on the waveform),
# so the mask probabilities are pushed up a bit from the 0.05 baseline
# used when it was paired with a waveform augmentation. Lower these
# back toward 0.05 if training loss stalls or WER gets worse, since
# masking too aggressively can make the task too hard to learn from.
model.config.apply_spec_augment = True
model.config.mask_time_prob = 0.08
model.config.mask_time_length = 10
model.config.mask_feature_prob = 0.08
model.config.mask_feature_length = 10

# Optional: freeze the encoder if your training set is very small
# (e.g. a few hours of audio). This forces the model to only adapt
# the decoder/language side, which is much less prone to overfitting
# on limited data. Uncomment if train loss still crashes to ~0 fast.
# model.freeze_encoder()

model.to(device)

# =========================================================
# NOTE ON AUGMENTATION
# No manual waveform augmentation (no speed/pitch/noise perturbation)
# in this version. Regularization comes entirely from SpecAugment,
# which is applied automatically inside the model on the extracted
# spectrogram features during training (see model.config settings
# above: apply_spec_augment, mask_time_prob/length,
# mask_feature_prob/length). Because it lives in the model's forward
# pass rather than in preprocessing, it's active during trainer.train()
# and automatically skipped during eval/generation - no extra code
# needed here to keep validation clean.
# =========================================================

# =========================================================
# PREPROCESSING
# =========================================================
def prepare_dataset_train(batch):
    audio = batch["audio"]

    # No waveform-level augmentation here - SpecAugment (applied inside
    # the model, see model.config above) is the sole regularizer.
    batch["input_features"] = processor.feature_extractor(
        audio["array"],
        sampling_rate=audio["sampling_rate"]
    ).input_features[0]

    batch["labels"] = processor.tokenizer(batch["text"]).input_ids

    return batch

def prepare_dataset_eval(batch):
    audio = batch["audio"]

    batch["input_features"] = processor.feature_extractor(
        audio["array"],
        sampling_rate=audio["sampling_rate"]
    ).input_features[0]

    batch["labels"] = processor.tokenizer(batch["text"]).input_ids

    return batch

train_ds = train_ds.map(
    prepare_dataset_train,
    remove_columns=train_ds.column_names,
    num_proc=1
)

valid_ds = valid_ds.map(
    prepare_dataset_eval,
    remove_columns=valid_ds.column_names,
    num_proc=1
)

# =========================================================
# DATA COLLATOR
# =========================================================
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features):
        input_features = [{"input_features": f["input_features"]} for f in features]

        batch = self.processor.feature_extractor.pad(
            input_features,
            return_tensors="pt"
        )

        label_features = [{"input_ids": f["labels"]} for f in features]

        labels_batch = self.processor.tokenizer.pad(
            label_features,
            return_tensors="pt"
        )

        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1),
            -100
        )

        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor)

# =========================================================
# WER + CER METRICS
# =========================================================
wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str = processor.tokenizer.batch_decode(
        pred_ids,
        skip_special_tokens=True
    )

    label_str = processor.tokenizer.batch_decode(
        label_ids,
        skip_special_tokens=True
    )

    wer = wer_metric.compute(
        predictions=pred_str,
        references=label_str
    )

    cer = cer_metric.compute(
        predictions=pred_str,
        references=label_str
    )

    return {
        "wer": wer,
        "cer": cer
    }

# =========================================================
# TRAINING CONFIG (FIXED - anti-overfitting)
# =========================================================
training_args = Seq2SeqTrainingArguments(
    output_dir="SPCS/models_result/whisper_specaugment",

    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,   # effective batch size 8

    learning_rate=1e-5,
    lr_scheduler_type="linear",
    warmup_ratio=0.1,               # smooth ramp-up instead of jumping straight to peak LR
    weight_decay=0.01,              # penalizes large weights, directly fights overfitting

    num_train_epochs=10,            # kept high on purpose - early stopping will cut it short
    max_grad_norm=1.0,

    fp16=torch.cuda.is_available(),
    gradient_checkpointing=True,

    eval_strategy="epoch",
    save_strategy="epoch",

    logging_steps=25,
    save_total_limit=3,

    predict_with_generate=True,
    generation_max_length=225,

    # Keep the checkpoint with the best validation WER, not the last one.
    # This alone fixes most of the damage from overfitting: even if
    # training runs long, you deploy the epoch that generalized best.
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,

    # NOTE: safe_serialization is handled manually at save time below
    # (this transformers version doesn't accept save_safetensors here).

    report_to="none"
)

# =========================================================
# TRAINER
# =========================================================
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=valid_ds,
    data_collator=data_collator,
    processing_class=processor.feature_extractor,
    compute_metrics=compute_metrics,
    # Stops training if validation WER hasn't improved for 3 straight
    # evals - prevents the "train loss -> 0, val loss flat" pattern
    # you saw from running all the way to epoch 10.
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

# =========================================================
# TRAIN
# =========================================================
trainer.train()

# =========================================================
# SAVE MODEL (this saves the BEST checkpoint, thanks to
# load_best_model_at_end=True above)
# =========================================================
save_path = "SPCS/models_result/whisper_specaugment"

model.tie_weights()  # make sure proj_out is explicitly linked to the embedding weights

# Saving directly via model.save_pretrained (rather than trainer.save_model)
# so we can pass safe_serialization=False - this avoids the "missing keys:
# proj_out.weight" warning caused by Whisper's tied embedding/output layer
# being dropped from safetensors checkpoints on save.
model.save_pretrained(save_path, safe_serialization=False)
processor.save_pretrained(save_path)

print("\nTraining complete.")
print("Best model (by validation WER) saved to:", save_path)

Enter your Hugging Face token:  ········


Using device: cuda
Training samples: 9908
Validation samples: 1239


Loading weights:   0%|          | 0/167 [00:00<?, ?it/s]

Map:   0%|          | 0/9908 [00:00<?, ? examples/s]

Map:   0%|          | 0/1239 [00:00<?, ? examples/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['proj_out.weight'].


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Training complete.
Best model (by validation WER) saved to: SPCS/models_result/whisper_specaugment


In [ ]:
-------------------------------after this--------------------

In [5]:
# =========================================================
# INSTALL DEPENDENCIES
# =========================================================
!pip install -q transformers datasets evaluate jiwer accelerate soundfile huggingface_hub librosa

# =========================================================
# IMPORTS
# =========================================================
import os
import random
import numpy as np
import torch
import pandas as pd
import evaluate
import librosa

from huggingface_hub import login

from dataclasses import dataclass
from typing import Any

from datasets import Dataset, Audio
from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    EarlyStoppingCallback,
)

# =========================================================
# HF LOGIN
# Prompts you to paste your token at runtime (hidden input, like a
# PIN) instead of hardcoding it in the script.
# =========================================================
from getpass import getpass

login(getpass("Enter your Hugging Face token: "))

# =========================================================
# DEVICE
# =========================================================
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# =========================================================
# PATHS
# =========================================================
train_csv = "SPCS/manifests/train.csv"
valid_csv = "SPCS/manifests/valid.csv"
test_csv = "SPCS/manifests/test.csv"

# =========================================================
# LOAD DATA
# =========================================================
train_df = pd.read_csv(train_csv)
valid_df = pd.read_csv(valid_csv)

print("Training samples:", len(train_df))
print("Validation samples:", len(valid_df))

# =========================================================
# DATASET CONVERSION
# =========================================================
train_ds = Dataset.from_pandas(train_df)
valid_ds = Dataset.from_pandas(valid_df)

train_ds = train_ds.cast_column("audio", Audio(sampling_rate=16000))
valid_ds = valid_ds.cast_column("audio", Audio(sampling_rate=16000))

# =========================================================
# MODEL + PROCESSOR
# Switch model_name to "openai/whisper-tiny" if small keeps
# overfitting hard even after the regularization changes below.
# =========================================================
model_name = "openai/whisper-tiny"

processor = WhisperProcessor.from_pretrained(model_name, task="transcribe")
model = WhisperForConditionalGeneration.from_pretrained(model_name)

model.generation_config.forced_decoder_ids = None
model.generation_config.suppress_tokens = []
model.generation_config.language = None
model.generation_config.task = "transcribe"

# ---- Regularization additions ----
# SpecAugment: randomly masks time/frequency bands in the input
# spectrogram during training, forcing the model to not rely on
# any single narrow slice of audio. Cheap and effective against
# overfitting on small speech datasets. This is the ONLY augmentation
# in this version (no speed/pitch/noise perturbation on the waveform),
# so the mask probabilities are pushed up a bit from the 0.05 baseline
# used when it was paired with a waveform augmentation. Lower these
# back toward 0.05 if training loss stalls or WER gets worse, since
# masking too aggressively can make the task too hard to learn from.
model.config.apply_spec_augment = True
model.config.mask_time_prob = 0.08
model.config.mask_time_length = 10
model.config.mask_feature_prob = 0.08
model.config.mask_feature_length = 10

# Optional: freeze the encoder if your training set is very small
# (e.g. a few hours of audio). This forces the model to only adapt
# the decoder/language side, which is much less prone to overfitting
# on limited data. Uncomment if train loss still crashes to ~0 fast.
# model.freeze_encoder()

model.to(device)

# =========================================================
# NOTE ON AUGMENTATION
# No manual waveform augmentation (no speed/pitch/noise perturbation)
# in this version. Regularization comes entirely from SpecAugment,
# which is applied automatically inside the model on the extracted
# spectrogram features during training (see model.config settings
# above: apply_spec_augment, mask_time_prob/length,
# mask_feature_prob/length). Because it lives in the model's forward
# pass rather than in preprocessing, it's active during trainer.train()
# and automatically skipped during eval/generation - no extra code
# needed here to keep validation clean.
# =========================================================

# =========================================================
# PREPROCESSING
# =========================================================
def prepare_dataset_train(batch):
    audio = batch["audio"]

    # No waveform-level augmentation here - SpecAugment (applied inside
    # the model, see model.config above) is the sole regularizer.
    batch["input_features"] = processor.feature_extractor(
        audio["array"],
        sampling_rate=audio["sampling_rate"]
    ).input_features[0]

    batch["labels"] = processor.tokenizer(batch["text"]).input_ids

    return batch

def prepare_dataset_eval(batch):
    audio = batch["audio"]

    batch["input_features"] = processor.feature_extractor(
        audio["array"],
        sampling_rate=audio["sampling_rate"]
    ).input_features[0]

    batch["labels"] = processor.tokenizer(batch["text"]).input_ids

    return batch

train_ds = train_ds.map(
    prepare_dataset_train,
    remove_columns=train_ds.column_names,
    num_proc=1
)

valid_ds = valid_ds.map(
    prepare_dataset_eval,
    remove_columns=valid_ds.column_names,
    num_proc=1
)

# =========================================================
# DATA COLLATOR
# =========================================================
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features):
        input_features = [{"input_features": f["input_features"]} for f in features]

        batch = self.processor.feature_extractor.pad(
            input_features,
            return_tensors="pt"
        )

        label_features = [{"input_ids": f["labels"]} for f in features]

        labels_batch = self.processor.tokenizer.pad(
            label_features,
            return_tensors="pt"
        )

        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1),
            -100
        )

        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor)

# =========================================================
# WER + CER METRICS
# =========================================================
wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str = processor.tokenizer.batch_decode(
        pred_ids,
        skip_special_tokens=True
    )

    label_str = processor.tokenizer.batch_decode(
        label_ids,
        skip_special_tokens=True
    )

    wer = wer_metric.compute(
        predictions=pred_str,
        references=label_str
    )

    cer = cer_metric.compute(
        predictions=pred_str,
        references=label_str
    )

    return {
        "wer": wer,
        "cer": cer
    }

# =========================================================
# TRAINING CONFIG (FIXED - anti-overfitting)
# =========================================================
training_args = Seq2SeqTrainingArguments(
    output_dir="SPCS/models_result/whisper_specaugment",

    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,   # effective batch size 8

    learning_rate=1e-5,
    lr_scheduler_type="linear",
    warmup_ratio=0.1,               # smooth ramp-up instead of jumping straight to peak LR
    weight_decay=0.01,              # penalizes large weights, directly fights overfitting

    num_train_epochs=10,            # kept high on purpose - early stopping will cut it short
    max_grad_norm=1.0,

    fp16=torch.cuda.is_available(),
    gradient_checkpointing=True,

    eval_strategy="epoch",
    save_strategy="epoch",

    logging_steps=25,
    save_total_limit=3,

    predict_with_generate=True,
    generation_max_length=225,

    # Keep the checkpoint with the best validation WER, not the last one.
    # This alone fixes most of the damage from overfitting: even if
    # training runs long, you deploy the epoch that generalized best.
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,

    # NOTE: safe_serialization is handled manually at save time below
    # (this transformers version doesn't accept save_safetensors here).

    report_to="none"
)

# =========================================================
# TRAINER
# =========================================================
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=valid_ds,
    data_collator=data_collator,
    processing_class=processor.feature_extractor,
    compute_metrics=compute_metrics,
    # Stops training if validation WER hasn't improved for 3 straight
    # evals - prevents the "train loss -> 0, val loss flat" pattern
    # you saw from running all the way to epoch 10.
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

# =========================================================
# TRAIN
# =========================================================
trainer.train()

# =========================================================
# SAVE MODEL (this saves the BEST checkpoint, thanks to
# load_best_model_at_end=True above)
# =========================================================
save_path = "SPCS/models_result/whisper_specaugment"

model.tie_weights()  # make sure proj_out is explicitly linked to the embedding weights

# Saving directly via model.save_pretrained (rather than trainer.save_model)
# so we can pass safe_serialization=False - this avoids the "missing keys:
# proj_out.weight" warning caused by Whisper's tied embedding/output layer
# being dropped from safetensors checkpoints on save.
model.save_pretrained(save_path, safe_serialization=False)
processor.save_pretrained(save_path)

# =========================================================
# SAVE TRAINER STATE (training loss / val loss / wer / cer history)
# Writes trainer_state.json into save_path so the full log_history
# is reloadable later, instead of only living in checkpoint-N folders.
# =========================================================
trainer.save_state()

log_history = trainer.state.log_history
log_df = pd.DataFrame(log_history)
log_csv_path = os.path.join(save_path, "training_log_history.csv")
log_df.to_csv(log_csv_path, index=False)

print("\nTraining complete.")
print("Best model (by validation WER) saved to:", save_path)
print("Trainer state saved to:", save_path)
print("Training log history exported to:", log_csv_path)

# =========================================================
# TEST SET EVALUATION (NEW)
# Runs the best saved checkpoint on the held-out test set - data
# never seen during training or used for checkpoint selection.
# This gives the final, unbiased WER/CER for the SpecAugment model.
# =========================================================
test_df = pd.read_csv(test_csv)
print("\nTest samples:", len(test_df))

test_ds = Dataset.from_pandas(test_df)
test_ds = test_ds.cast_column("audio", Audio(sampling_rate=16000))
test_ds = test_ds.map(prepare_dataset_eval, remove_columns=test_ds.column_names, num_proc=1)

test_results = trainer.evaluate(eval_dataset=test_ds, metric_key_prefix="test")

print("\nTest set results:")
print(test_results)

# ---- Save test results to CSV ----
test_results_df = pd.DataFrame([test_results])
test_results_csv_path = os.path.join(save_path, "test_results.csv")
test_results_df.to_csv(test_results_csv_path, index=False)
print("Test results saved to:", test_results_csv_path)

Enter your Hugging Face token:  ········


Using device: cuda
Training samples: 9908
Validation samples: 1239


Loading weights:   0%|          | 0/167 [00:00<?, ?it/s]

Map:   0%|          | 0/9908 [00:00<?, ? examples/s]

Map:   0%|          | 0/1239 [00:00<?, ? examples/s]

...


Training complete.
Best model (by validation WER) saved to: SPCS/models_result/whisper_specaugment
Trainer state saved to: SPCS/models_result/whisper_specaugment
Training log history exported to: SPCS/models_result/whisper_specaugment/training_log_history.csv

Test samples: 1239


Map:   0%|          | 0/1239 [00:00<?, ? examples/s]

[transformers] early stopping required metric_for_best_model, but did not find eval_wer so early stopping is disabled


Training Loss,Validation Loss,Epoch,Wer,Cer
0.013412,0.009368,10,0.022124,0.021571



Test set results:
{'test_loss': 0.009368144907057285, 'test_wer': 0.022123893805309734, 'test_cer': 0.021570934502798872}
Test results saved to: SPCS/models_result/whisper_specaugment/test_results.csv


In [6]:
# =========================================================
# INSTALL DEPENDENCIES
# =========================================================
!pip install -q transformers datasets evaluate jiwer accelerate soundfile huggingface_hub librosa

# =========================================================
# IMPORTS
# =========================================================
import os
import random
import numpy as np
import torch
import pandas as pd
import evaluate
import librosa

from huggingface_hub import login

from dataclasses import dataclass
from typing import Any

from datasets import Dataset, Audio
from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    EarlyStoppingCallback,
)

# =========================================================
# HF LOGIN
# Prompts you to paste your token at runtime (hidden input, like a
# PIN) instead of hardcoding it in the script.
# =========================================================
from getpass import getpass

login(getpass("Enter your Hugging Face token: "))

# =========================================================
# DEVICE
# =========================================================
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# =========================================================
# PATHS
# =========================================================
train_csv = "SPCS/manifests/train.csv"
valid_csv = "SPCS/manifests/valid.csv"
test_csv = "SPCS/manifests/test.csv"

# =========================================================
# LOAD DATA
# =========================================================
train_df = pd.read_csv(train_csv)
valid_df = pd.read_csv(valid_csv)

print("Training samples:", len(train_df))
print("Validation samples:", len(valid_df))

# =========================================================
# DATASET CONVERSION
# =========================================================
train_ds = Dataset.from_pandas(train_df)
valid_ds = Dataset.from_pandas(valid_df)

train_ds = train_ds.cast_column("audio", Audio(sampling_rate=16000))
valid_ds = valid_ds.cast_column("audio", Audio(sampling_rate=16000))

# =========================================================
# MODEL + PROCESSOR
# Switch model_name to "openai/whisper-tiny" if small keeps
# overfitting hard even after the regularization changes below.
# =========================================================
model_name = "openai/whisper-tiny"

processor = WhisperProcessor.from_pretrained(model_name, task="transcribe")
model = WhisperForConditionalGeneration.from_pretrained(model_name)

model.generation_config.forced_decoder_ids = None
model.generation_config.suppress_tokens = []
model.generation_config.language = None
model.generation_config.task = "transcribe"

# ---- Regularization additions ----
# SpecAugment: randomly masks time/frequency bands in the input
# spectrogram during training, forcing the model to not rely on
# any single narrow slice of audio. Cheap and effective against
# overfitting on small speech datasets.
odel.config.apply_spec_augment = True
model.config.mask_time_prob = 0.05
model.config.mask_time_length = 10
model.config.mask_feature_prob = 0.05
model.config.mask_feature_length = 10

# Optional: freeze the encoder if your training set is very small
# (e.g. a few hours of audio). This forces the model to only adapt
# the decoder/language side, which is much less prone to overfitting
# on limited data. Uncomment if train loss still crashes to ~0 fast.
# model.freeze_encoder()

model.to(device)

# =========================================================
# NOISE AUGMENTATION
# Randomly adds low-level Gaussian noise to training audio before
# feature extraction. This is a standard, cheap augmentation for ASR
# that exposes the model to noisy input conditions, helping it
# generalize better and reducing overfitting on a small dataset.
# Applied to TRAINING data only - validation audio must stay
# unmodified so your WER/CER stay a fair, untouched measure of real
# performance.
# =========================================================
NOISE_FACTORS = [0.0, 0.002, 0.005, 0.01]  # 0.0 = unchanged, higher = more noise added

def apply_noise_augmentation(audio_array, sampling_rate):
    noise_factor = random.choice(NOISE_FACTORS)
    if noise_factor == 0.0:
        return audio_array
    audio_array = audio_array.astype(np.float32)
    # Scale noise relative to the signal's own amplitude so quiet and
    # loud clips get a comparable relative amount of noise.
    noise = np.random.randn(len(audio_array)).astype(np.float32)
    return audio_array + noise_factor * np.max(np.abs(audio_array)) * noise

# =========================================================
# PREPROCESSING
# =========================================================
def prepare_dataset_train(batch):
    audio = batch["audio"]

    audio_array = apply_noise_augmentation(audio["array"], audio["sampling_rate"])

    batch["input_features"] = processor.feature_extractor(
        audio_array,
        sampling_rate=audio["sampling_rate"]
    ).input_features[0]

    batch["labels"] = processor.tokenizer(batch["text"]).input_ids

    return batch

def prepare_dataset_eval(batch):
    audio = batch["audio"]

    batch["input_features"] = processor.feature_extractor(
        audio["array"],
        sampling_rate=audio["sampling_rate"]
    ).input_features[0]

    batch["labels"] = processor.tokenizer(batch["text"]).input_ids

    return batch

train_ds = train_ds.map(
    prepare_dataset_train,
    remove_columns=train_ds.column_names,
    num_proc=1
)

valid_ds = valid_ds.map(
    prepare_dataset_eval,
    remove_columns=valid_ds.column_names,
    num_proc=1
)

# =========================================================
# DATA COLLATOR
# =========================================================
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features):
        input_features = [{"input_features": f["input_features"]} for f in features]

        batch = self.processor.feature_extractor.pad(
            input_features,
            return_tensors="pt"
        )

        label_features = [{"input_ids": f["labels"]} for f in features]

        labels_batch = self.processor.tokenizer.pad(
            label_features,
            return_tensors="pt"
        )

        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1),
            -100
        )

        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor)

# =========================================================
# WER + CER METRICS
# =========================================================
wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str = processor.tokenizer.batch_decode(
        pred_ids,
        skip_special_tokens=True
    )

    label_str = processor.tokenizer.batch_decode(
        label_ids,
        skip_special_tokens=True
    )

    wer = wer_metric.compute(
        predictions=pred_str,
        references=label_str
    )

    cer = cer_metric.compute(
        predictions=pred_str,
        references=label_str
    )

    return {
        "wer": wer,
        "cer": cer
    }

# =========================================================
# TRAINING CONFIG (FIXED - anti-overfitting)
# =========================================================
training_args = Seq2SeqTrainingArguments(
    output_dir="SPCS/models_result/whisper_noise_aug",

    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,   # effective batch size 8

    learning_rate=1e-5,
    lr_scheduler_type="linear",
    warmup_ratio=0.1,               # smooth ramp-up instead of jumping straight to peak LR
    weight_decay=0.01,              # penalizes large weights, directly fights overfitting

    num_train_epochs=10,            # kept high on purpose - early stopping will cut it short
    max_grad_norm=1.0,

    fp16=torch.cuda.is_available(),
    gradient_checkpointing=True,

    eval_strategy="epoch",
    save_strategy="epoch",

    logging_steps=25,
    save_total_limit=3,

    predict_with_generate=True,
    generation_max_length=225,

    # Keep the checkpoint with the best validation WER, not the last one.
    # This alone fixes most of the damage from overfitting: even if
    # training runs long, you deploy the epoch that generalized best.
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,

    # NOTE: safe_serialization is handled manually at save time below
    # (this transformers version doesn't accept save_safetensors here).

    report_to="none"
)

# =========================================================
# TRAINER
# =========================================================
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=valid_ds,
    data_collator=data_collator,
    processing_class=processor.feature_extractor,
    compute_metrics=compute_metrics,
    # Stops training if validation WER hasn't improved for 3 straight
    # evals - prevents the "train loss -> 0, val loss flat" pattern
    # you saw from running all the way to epoch 10.
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

# =========================================================
# TRAIN
# =========================================================
trainer.train()

# =========================================================
# SAVE MODEL (this saves the BEST checkpoint, thanks to
# load_best_model_at_end=True above)
# =========================================================
save_path = "SPCS/models_result/whisper_noise_aug"

model.tie_weights()  # make sure proj_out is explicitly linked to the embedding weights

# Saving directly via model.save_pretrained (rather than trainer.save_model)
# so we can pass safe_serialization=False - this avoids the "missing keys:
# proj_out.weight" warning caused by Whisper's tied embedding/output layer
# being dropped from safetensors checkpoints on save.
model.save_pretrained(save_path, safe_serialization=False)
processor.save_pretrained(save_path)

# =========================================================
# SAVE TRAINER STATE (training loss / val loss / wer / cer history)
# Writes trainer_state.json into save_path so the full log_history
# is reloadable later, instead of only living in checkpoint-N folders.
# =========================================================
trainer.save_state()

log_history = trainer.state.log_history
log_df = pd.DataFrame(log_history)
log_csv_path = os.path.join(save_path, "training_log_history.csv")
log_df.to_csv(log_csv_path, index=False)

print("\nTraining complete.")
print("Best model (by validation WER) saved to:", save_path)
print("Trainer state saved to:", save_path)
print("Training log history exported to:", log_csv_path)

# =========================================================
# TEST SET EVALUATION (NEW)
# Runs the best saved checkpoint on the held-out test set - data
# never seen during training or used for checkpoint selection.
# This gives the final, unbiased WER/CER for the noise-augmented model.
# =========================================================
test_df = pd.read_csv(test_csv)
print("\nTest samples:", len(test_df))

test_ds = Dataset.from_pandas(test_df)
test_ds = test_ds.cast_column("audio", Audio(sampling_rate=16000))
test_ds = test_ds.map(prepare_dataset_eval, remove_columns=test_ds.column_names, num_proc=1)

test_results = trainer.evaluate(eval_dataset=test_ds, metric_key_prefix="test")

print("\nTest set results:")
print(test_results)

# ---- Save test results to CSV ----
test_results_df = pd.DataFrame([test_results])
test_results_csv_path = os.path.join(save_path, "test_results.csv")
test_results_df.to_csv(test_results_csv_path, index=False)
print("Test results saved to:", test_results_csv_path)

Enter your Hugging Face token:  ········


Using device: cuda
Training samples: 9908
Validation samples: 1239


Loading weights:   0%|          | 0/167 [00:00<?, ?it/s]

Map:   0%|          | 0/9908 [00:00<?, ? examples/s]

Map:   0%|          | 0/1239 [00:00<?, ? examples/s]

...


Training complete.
Best model (by validation WER) saved to: SPCS/models_result/whisper_noise_aug
Trainer state saved to: SPCS/models_result/whisper_noise_aug
Training log history exported to: SPCS/models_result/whisper_noise_aug/training_log_history.csv

Test samples: 1239


Map:   0%|          | 0/1239 [00:00<?, ? examples/s]

[transformers] early stopping required metric_for_best_model, but did not find eval_wer so early stopping is disabled


Training Loss,Validation Loss,Epoch,Wer,Cer
0.020245,0.013267,10,0.018990,0.019432



Test set results:
{'test_loss': 0.01326702255755663, 'test_wer': 0.01898967551622419, 'test_cer': 0.01943166827111634}
Test results saved to: SPCS/models_result/whisper_noise_aug/test_results.csv
